In [34]:


CLASS_MAP = {
    0: 'airplane', 1: 'bathtub', 2: 'bed', 3: 'bench', 4: 'bookshelf',
    5: 'bottle', 6: 'bowl', 7: 'car', 8: 'chair', 9: 'cone',
    10: 'cup', 11: 'curtain', 12: 'desk', 13: 'door', 14: 'dresser',
    15: 'flower_pot', 16: 'glass_box', 17: 'guitar', 18: 'keyboard', 19: 'lamp',
    20: 'laptop', 21: 'mantel', 22: 'monitor', 23: 'night_stand', 24: 'person',
    25: 'piano', 26: 'plant', 27: 'radio', 28: 'range_hood', 29: 'sink',
    30: 'sofa', 31: 'stairs', 32: 'stool', 33: 'table', 34: 'tent',
    35: 'toilet', 36: 'tv_stand', 37: 'vase', 38: 'wardrobe', 39: 'xbox'
}

In [35]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

def apply_convolution_layer(x, filters):
    x = layers.Conv1D(filters=filters, kernel_size=1, padding="valid")(x)
    x = layers.BatchNormalization(momentum=0.0)(x)
    return layers.ReLU()(x)

def apply_dense_layer(x, units, dropout_rate=0.3):
    x = layers.Dense(units)(x)
    x = layers.BatchNormalization(momentum=0.0)(x)
    return layers.Activation("relu")(x)

def transform_net(inputs, output_channels):
    x = apply_convolution_layer(inputs, 64)
    x = apply_convolution_layer(x, 128)
    x = apply_convolution_layer(x, 1024)
    x = layers.GlobalMaxPooling1D()(x)
    x = apply_dense_layer(x, 512)
    x = apply_dense_layer(x, 256)
    
    bias = tf.keras.initializers.Constant(np.eye(output_channels).flatten())
    x = layers.Dense(
        output_channels * output_channels,
        kernel_initializer="zeros",
        bias_initializer=bias,)(x)
    
    x = layers.Reshape((output_channels, output_channels))(x)
    return layers.Dot(axes=(2, 1))([inputs, x])


points_num = 2048
classes_num = 40

inputs = keras.Input(shape=(points_num, 3))


x = transform_net(inputs, output_channels=3)
x = apply_convolution_layer(x, filters=64)
x = apply_convolution_layer(x, filters=64)
x = transform_net(x, output_channels=64)
x = apply_convolution_layer(x, filters=64)
x = apply_convolution_layer(x, filters=128)
x = apply_convolution_layer(x, filters=1024)
x = layers.GlobalMaxPooling1D()(x)

x = apply_dense_layer(x, units=512)
x = layers.Dropout(0.3)(x)
x = apply_dense_layer(x, units=256)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(classes_num, activation="softmax")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="pointnet")


try:
    model.load_weights("pointnet_modelnet_weights_tf_dim_ordering_tf_kernels.h5")
    print("Готовая модель успешно загружена!")
except Exception as e:
    print(f"Ошибка при загрузке весов: {e}")


Готовая модель успешно загружена!


In [37]:
import trimesh
import numpy as np
import tensorflow as tf

def predict_single_object(model, file_path, points_num=2048, noise_level=0.1):

    mesh = trimesh.load(file_path, force='mesh')
    points = mesh.sample(points_num)
    

    points -= np.mean(points, axis=0)
    points /= np.max(np.linalg.norm(points, axis=1))
    points = points[:, [0, 2, 1]]
    

    input_original = np.expand_dims(points, axis=0).astype('float32')
    preds_orig = model.predict(input_original)
    idx_orig = np.argmax(preds_orig)
    conf_orig = preds_orig[0][idx_orig]
    

    
    noise = np.random.normal(0, noise_level, size=points.shape)
    points_noisy = points + noise
    
    input_noisy = np.expand_dims(points_noisy, axis=0).astype('float32')
    preds_noisy = model.predict(input_noisy)
    idx_noisy = np.argmax(preds_noisy)
    conf_noisy = preds_noisy[0][idx_noisy]
    

    print(f"Оригинал: {CLASS_MAP[idx_orig]} ({conf_orig:.4%})")
    print(f"С шумом({noise_level}):  {CLASS_MAP[idx_noisy]} ({conf_noisy:.4%})")
        

my_file = "data/guitar_0156.off" 

predict_single_object(model, my_file)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
Оригинал: guitar (99.9830%)
С шумом(0.1):  piano (44.9604%)
